## Race Conditions



Eine Race Condition entsteht, wenn mindestens zwei Threads gleichzeitig auf eine **gemeinsame Ressource** zugreifen 
und diese **verändern**, ohne Synchronisation. 
=> Ergebnis ist abhängig vom Zeitpunkt des Zugriffs und kann **inkonsistent** sein

### Arten von Race Conditions

1. **Read-Modify-Write**  
   - Ein Thread liest, verändert und schreibt einen Wert  
   - Währenddessen schreibt ein anderer Thread denselben Wert => inkonsistener Wert

2. **Check-Then-Act**  
   - Ein Thread prüft eine Bedingung und handelt danach  
   - Ein anderer Thread ändert diese Bedingung zwischenzeitlich  
   - Beispiel: Zwei Threads sehen eine leere Liste und fügen gleichzeitig ein Element hinzu => doppelte Einträge



In [ ]:
//Read Modify Write Beispiel 
var counter = 0

class MyThread extends Thread {
  override def run(): Unit = {
    for (i <- 1 to 10000) {
        counter = counter +1 
        println(counter)
    }
  }
}

val t1 = new MyThread()
val t2 = new MyThread()
t1.start()
t2.start()

1
2
1
4
5
3
7
6
9
8
10
12
13
11
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
14
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
46
64
65
66
67
68
69
70
71
72
73
74
75
63
77
78
79
80
76
82
83
84
81
86
87
85
89
90
91
88
92
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
93
111
112
113
114
115
116
117
118
119
120
121
110
123
124
125
126
122
128
129
127
130
132
133
134
135
136
137
138
139
140
141
142
131
144
145
143
147
148
149
150
151
152
153
154
155
156
157
158
159
146
161
162
163
164
165
166
167
168
169
170
171
172
173
174
160
175
177
178
179
180
181
182
176
184
185
186
183
188
189
190
191
192


counter: Int = 14
defined class MyThread
t1: MyThread = Thread[#380,Thread-307,5,main]
t2: MyThread = Thread[#381,Thread-308,5,main]

In [3]:
println(counter) //Ergebnis ungewiss

2


In [ ]:
//Check-Then-Act Beispiel
class Konto(var kontostand: Int) {

  def abheben(betrag: Int): Boolean = {
    if (kontostand >= betrag) {
      Thread.sleep(100)
      kontostand -= betrag
      true
    } else {
      false
    }
  }
}
val konto = new Konto(100)
val threads =
  (1 to 10).map { i =>
    new Thread(() => {
      val ok = konto.abheben(20)
      println(s"$i -> $ok")
    })
  }
threads.foreach(_.start())
threads.foreach(_.join())
println(s"Ende: ${konto.kontostand}")

2 -> true
1 -> true
4 -> true
9 -> true
8 -> true
3 -> true
7 -> true
6 -> true
5 -> true
10 -> true
Ende: -40


defined class Konto
konto: Konto = ammonite.$sess.cmd77$Helper$Konto@6ca38bde
threads: IndexedSeq[Thread] = Vector(
  Thread[#428,Thread-355,5,],
  Thread[#429,Thread-356,5,],
  Thread[#430,Thread-357,5,],
  Thread[#431,Thread-358,5,],
  Thread[#432,Thread-359,5,],
  Thread[#433,Thread-360,5,],
  Thread[#434,Thread-361,5,],
  Thread[#435,Thread-362,5,],
  Thread[#436,Thread-363,5,],
  Thread[#437,Thread-364,5,]
)

### Kritischer Bereich (Critical Section)

- Abschnitt eines Programms, wo potentielle Race Conditions entstehen.
- Im Abschnitt wird auf gemeinsam genutzte Ressourcen zugegriffen (z. B. Variablen).
- Wird typischerweise mit **Monitoren** oder **Semaphoren** geschützt.

### Atomare Klassen (z.B. AtomicInteger) 
- nutzen low-level Mechanismen, wie der sogenannte Compare-and-Swap (CAS)  
- ermöglichen einfache thread-sichere Operationen wie .incrementAndGet() auf einzelnen Variablen ohne explizite Sperren (Locks)
- Besonders geeignet für einfache Zähler, Flags oder Referenzen, bei denen nur einzelne atomare Operationen benötigt werden.

In [ ]:
import java.util.concurrent.atomic.AtomicInteger

val counter = new AtomicInteger(0)

class MyThread extends Thread {
  override def run(): Unit = {
    for (_ <- 1 to 1000) {
      counter.incrementAndGet
    }
  }
}
val t1 = new MyThread()
val t2 = new MyThread()
t1.start()
t2.start()

import java.util.concurrent.atomic.AtomicInteger


counter: AtomicInteger = 2000
defined class MyThread
t1: MyThread = Thread[#467,Thread-392,5,]
t2: MyThread = Thread[#468,Thread-393,5,]

In [79]:
println(counter) // Ergebnis 2000

2000


### 1. Monitor mit .synchronized()
- der Monitor garantiert dass nur ein Thread den kritischen Bereich betreten kann
- Jede Referenzvariable kann als Monitor genutzt werden
- zusätzlich ermöglicht der Monitor Thread-Koordination über wait(), notify() und notifyAll()




In [ ]:
val lock = new Object()
var counter = 0

class MyThread extends Thread {
  override def run(): Unit = {
    for (_ <- 1 to 1000) {
        counter += 1
            lock.wait()
    }
  }
}
val t1 = new MyThread()
val t2 = new MyThread()
t1.start()
t2.start()

lock: Object = java.lang.Object@3f0bcb5
counter: Int = 2
defined class MyThread
t1: MyThread = Thread[#76,Thread-17,5,main]
t2: MyThread = Thread[#77,Thread-18,5,main]

In [2]:
println(counter)

2


### 3. Semaphore
- die Semaphore garantiert dass nur eine bestimmte Anzahl an Thread den kritischen Bereich betreten kann-
- new `Semaphore(3)` bedeutet: Maximal 3 Threads dürfen in den kritischen Bereich
- der kritische Bereich wird gekennzeichnet über semaphore.aquire() und semaphore.release


In [ ]:
import java.util.concurrent.Semaphore
import scala.util.Random
import scala.collection.mutable.ListBuffer

class Parkplatz(maxPlaetze: Int) {
  private val sem = new Semaphore(maxPlaetze)

  def parken(autoNummer: Int): Boolean = {
    try {
      sem.acquire()
      println("Auto " + autoNummer + " parkt.")
      Thread.sleep(1000 + (Random.nextDouble() * 2000).toInt)
      println("Auto " + autoNummer + " fährt weg.")
      true
    } catch {
      case _: InterruptedException =>
        println("Auto " + autoNummer + " wurde unterbrochen.")
        false
    } finally {
      sem.release()
    }
  }
}

class Auto(val nummer: Int, parkplatz: Parkplatz) extends Thread {
  override def run(): Unit = {
    while (!parkplatz.parken(nummer)) {
      try {
        Thread.sleep(500)
      } catch {
        case _: InterruptedException =>
      }
    }
  }
}
var listAuto =new  ListBuffer[Auto]()

val parkplatz = new Parkplatz(3)
    for (i <- 1 to 6) {
        val auto =  new Auto(i, parkplatz)
        auto.start()
        listAuto.addOne( auto)
    }
     for (auto <- listAuto) {
        auto.join()
     }

Auto 1 parkt.
Auto 2 parkt.
Auto 3 parkt.
Auto 2 fährt weg.
Auto 4 parkt.
Auto 1 fährt weg.
Auto 5 parkt.
Auto 3 fährt weg.
Auto 6 parkt.
Auto 5 fährt weg.
Auto 4 fährt weg.
Auto 6 fährt weg.


import java.util.concurrent.Semaphore

import scala.util.Random

import scala.collection.mutable.ListBuffer


defined class Parkplatz
defined class Auto
listAuto: ListBuffer[Auto] = ListBuffer(
  Thread[#279,Thread-214,5,],
  Thread[#280,Thread-215,5,],
  Thread[#281,Thread-216,5,],
  Thread[#282,Thread-217,5,],
  Thread[#283,Thread-218,5,],
  Thread[#284,Thread-219,5,]
)
parkplatz: Parkplatz = ammonite.$sess.cmd6$Helper$Parkplatz@62030266